# All-topics methodology test (Brexit + Covid + energy_crisis) — standalone



**all combined into one pipeline per run:**



In [1]:
!pip install -q transformers torch scikit-learn statsmodels pandas tqdm requests

In [2]:
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = "/content/drive/MyDrive/brexit_covid_datasets"
print("Reading from:", DATA_DIR)

Mounted at /content/drive
Reading from: /content/drive/MyDrive/brexit_covid_datasets


In [3]:
import pandas as pd
import numpy as np
import re
from datetime import datetime

# ---- CONFIG ----
ECON_INDICATOR_PATH = f"{DATA_DIR}/economic_indicator.csv"

OUTLETS = ["bbc", "dailymail", "guardian"]
TOPICS = ["brexit", "covid", "energy_crisis"]

AGG_FREQ = "MS"
LSTM_LOOKBACK = 3
TEST_FRACTION = 0.2
RANDOM_STATE = 42

# same keyword lists used to originally scrape each topic — reused here for
# aspect-based sentiment (extracting only the sentences that mention them)
TOPIC_KEYWORDS = {
    "brexit": ["Brexit", "EU referendum", "leave the EU", "Article 50",
               "Brexit deal", "no-deal Brexit", "Brexit negotiations", "Brexit vote"],
    "covid": ["coronavirus", "COVID-19", "COVID", "pandemic",
              "lockdown", "NHS coronavirus", "vaccine rollout", "Covid restrictions"],
    "energy_crisis": ["energy crisis", "energy price cap", "Ofgem price cap",
                       "energy supplier collapse", "Bulb Energy", "energy price guarantee",
                       "Energy Bills Support Scheme", "gas prices", "energy bills",
                       "cost of living payment", "windfall tax", "fuel poverty"],
}

# both recession windows are testable here, unlike the energy_crisis-only
# notebook — Brexit and Covid data both reach back before 2020
RECESSION_WINDOWS = [
    {"name": "2020 COVID recession", "start": pd.Timestamp("2020-01-01"), "end": pd.Timestamp("2020-06-30")},
    {"name": "2023 H2 recession",    "start": pd.Timestamp("2023-07-01"), "end": pd.Timestamp("2023-12-31")},
]
LEADUP_MONTHS = 6
CALM_COMPARISON_MONTHS = 6

# FRED public CSV endpoint — no API key needed for this
FRED_SERIES = {
    "unemployment": "LRHUTTTTGBM156S",   # UK unemployment rate, monthly, OECD/FRED
    "cpi": "GBRCPIALLMINMEI",            # UK CPI all items, monthly, OECD/FRED
}

## Load articles per (outlet, topic) — handles both file naming conventions

In [4]:
def load_outlet_topic(outlet, topic):
    if topic == "energy_crisis":
        path = f"{DATA_DIR}/{outlet}_energy_crisis_TEST.csv"
    else:
        path = f"{DATA_DIR}/{outlet}_brexit_covid_text_FINAL.csv"
    df = pd.read_csv(path)
    df = df[df["topic"] == topic].copy()
    df["publish_date"] = pd.to_datetime(df["publish_date"], errors="coerce", utc=True).dt.tz_localize(None)
    df = df.dropna(subset=["publish_date", "text"]).sort_values("publish_date").reset_index(drop=True)
    return df

datasets = {}
for outlet in OUTLETS:
    for topic in TOPICS:
        key = (outlet, topic)
        try:
            df = load_outlet_topic(outlet, topic)
            datasets[key] = df
            print(f"{outlet}/{topic}: {len(df)} articles, {df['publish_date'].min()} to {df['publish_date'].max()}")
        except FileNotFoundError:
            print(f"{outlet}/{topic}: FILE NOT FOUND")

bbc/brexit: 794 articles, 2012-06-17 00:00:00 to 2026-07-21 00:00:00
bbc/covid: 790 articles, 2020-01-10 00:00:00 to 2026-08-08 00:00:00
bbc/energy_crisis: 800 articles, 2021-08-06 00:00:00 to 2026-08-12 00:00:00
dailymail/brexit: 799 articles, 2012-01-28 00:00:00 to 2026-03-16 00:00:00
dailymail/covid: 798 articles, 2020-01-05 00:00:00 to 2026-03-16 00:00:00
dailymail/energy_crisis: 799 articles, 2021-08-07 00:00:00 to 2026-03-16 00:00:00
guardian/brexit: 800 articles, 2012-02-10 21:00:04 to 2026-08-04 04:00:12
guardian/covid: 800 articles, 2020-03-23 20:41:20 to 2026-08-08 20:03:34
guardian/energy_crisis: 800 articles, 2021-08-03 05:31:14 to 2026-08-13 14:54:03


## Macro data — GDP (existing) + unemployment + CPI (pulled live from FRED)

In [5]:
import requests
from io import StringIO

econ = pd.read_csv(ECON_INDICATOR_PATH)
econ["date"] = pd.to_datetime(econ["date"])
econ = econ.set_index("date").resample(AGG_FREQ).mean().rename(columns={"value": "econ_value"})
print(f"GDP indicator: {econ.index.min()} to {econ.index.max()}, {len(econ)} months")

fred_ids = ",".join(FRED_SERIES.values())
resp = requests.get(f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={fred_ids}", timeout=20)
fred_df = pd.read_csv(StringIO(resp.text))
fred_df["observation_date"] = pd.to_datetime(fred_df["observation_date"])
fred_df = fred_df.set_index("observation_date").rename(columns={
    FRED_SERIES["unemployment"]: "unemployment",
    FRED_SERIES["cpi"]: "cpi",
})
fred_df = fred_df.resample(AGG_FREQ).mean()
print(f"FRED macro data: {fred_df.index.min()} to {fred_df.index.max()}, {len(fred_df)} months")
print(fred_df.tail())

macro = econ.join(fred_df, how="left")
macro["growth"] = macro["econ_value"].pct_change() * 100
macro[["unemployment", "cpi"]] = macro[["unemployment", "cpi"]].ffill()  # macro data often reports with a short lag
print(f"\nCombined macro table: {len(macro)} months")
print(macro.tail())

GDP indicator: 1997-01-01 00:00:00 to 2026-05-01 00:00:00, 353 months
FRED macro data: 1955-01-01 00:00:00 to 2026-04-01 00:00:00, 856 months
                  unemployment  cpi
observation_date                   
2025-12-01                 5.2  NaN
2026-01-01                 4.9  NaN
2026-02-01                 5.0  NaN
2026-03-01                 4.9  NaN
2026-04-01                 4.9  NaN

Combined macro table: 353 months
            econ_value  unemployment    cpi    growth
date                                                 
2026-01-01       102.3           4.9  136.1  0.000000
2026-02-01       102.8           5.0  136.1  0.488759
2026-03-01       103.2           4.9  136.1  0.389105
2026-04-01       103.1           4.9  136.1 -0.096899
2026-05-01       103.2           4.9  136.1  0.096993


## Sentiment scoring — three methods (whole-article / aspect-based / ensemble)

In [6]:
from transformers import pipeline

finbert = pipeline(
    "sentiment-analysis",
    model="ProsusAI/finbert",
    tokenizer="ProsusAI/finbert",
    truncation=True,
    max_length=512,
    model_kwargs={"use_safetensors": False},
)

# second, general-purpose model for the ensemble method — deliberately NOT
# finance-tuned, so it brings a different bias than FinBERT rather than
# agreeing with it by construction
distilbert = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    truncation=True,
    max_length=512,
)

def _signed(results):
    out = []
    for r in results:
        label, score = r["label"].lower(), r["score"]
        out.append(score if label in ("positive", "pos") else (-score if label in ("negative", "neg") else 0.0))
    return out

def score_batch(model, texts, batch_size=16):
    scores = []
    for i in range(0, len(texts), batch_size):
        batch = [t[:2000] if isinstance(t, str) else "" for t in texts[i:i+batch_size]]
        scores.extend(_signed(model(batch)))
    return scores

def whole_article_scores(texts):
    """Method 1 (baseline) — same as the main/energy_crisis notebooks: FinBERT on the whole article."""
    return score_batch(finbert, texts)

def _split_sentences(text):
    # simple, dependency-free sentence split — good enough for keyword matching,
    # doesn't need to be linguistically perfect
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", str(text)) if s.strip()]

def aspect_based_scores(texts, keywords):
    """Method 2 — score only the sentences that mention the topic's own keywords,
    instead of the whole article, per Barbaglia et al. (2023)."""
    keywords_lower = [k.lower() for k in keywords]
    aspect_texts = []
    for t in texts:
        sentences = _split_sentences(t)
        matched = [s for s in sentences if any(k in s.lower() for k in keywords_lower)]
        # fall back to the first 2 sentences if nothing matched, rather than
        # scoring an empty string
        aspect_texts.append(" ".join(matched) if matched else " ".join(sentences[:2]))
    return score_batch(finbert, aspect_texts)

def ensemble_scores(texts):

    s1 = score_batch(finbert, texts)
    s2 = score_batch(distilbert, texts)
    return [(a + b) / 2 for a, b in zip(s1, s2)]

SENTIMENT_METHODS = {
    "whole_article": lambda texts, topic: whole_article_scores(texts),
    "aspect_based": lambda texts, topic: aspect_based_scores(texts, TOPIC_KEYWORDS[topic]),
    "ensemble": lambda texts, topic: ensemble_scores(texts),
}

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

... (5 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

## Models — Ridge, multivariate LSTM, naive + ARIMA baselines

In [7]:
import torch
import torch.nn as nn
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from statsmodels.tsa.arima.model import ARIMA

RIDGE_ALPHA_GRID = (0.1, 0.3, 1.0, 3.0, 10.0, 30.0)

def tune_ridge_alpha(X_train, y_train, alphas=RIDGE_ALPHA_GRID, min_months_for_cv=24):
    n = len(y_train)
    if n < min_months_for_cv:
        return 1.0, {1.0: None}
    n_splits = min(5, max(2, n // 12))
    tscv = TimeSeriesSplit(n_splits=n_splits)
    scores = {}
    for alpha in alphas:
        fold_rmses = []
        for tr_idx, val_idx in tscv.split(X_train):
            if len(tr_idx) < 3 or len(val_idx) < 1:
                continue
            m = Ridge(alpha=alpha).fit(X_train[tr_idx], y_train[tr_idx])
            pred = m.predict(X_train[val_idx])
            fold_rmses.append(mean_squared_error(y_train[val_idx], pred) ** 0.5)
        scores[alpha] = np.mean(fold_rmses) if fold_rmses else np.inf
    best_alpha = min(scores, key=scores.get)
    return best_alpha, scores

def evaluate(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    direction_match = np.sign(y_true) == np.sign(y_pred)
    return {
        "rmse": mean_squared_error(y_true, y_pred) ** 0.5,
        "mae": mean_absolute_error(y_true, y_pred),
        "r2": r2_score(y_true, y_pred) if len(y_true) > 1 else float("nan"),
        "directional_accuracy": direction_match.mean(),
    }

class SentimentLSTM(nn.Module):
    """input_size is now the number of feature columns (multivariate), not fixed to 1."""
    def __init__(self, input_size=1, hidden_size=16):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze(-1)

def train_lstm(X_train, y_train, input_size, epochs=150, lr=0.01, hidden_size=16, patience=15, val_fraction=0.2):
    torch.manual_seed(RANDOM_STATE)
    n = len(y_train)
    n_val = max(1, int(n * val_fraction)) if n >= 10 else 0
    if n_val > 0:
        X_fit, y_fit = X_train[:-n_val], y_train[:-n_val]
        X_val, y_val = X_train[-n_val:], y_train[-n_val:]
    else:
        X_fit, y_fit = X_train, y_train
        X_val, y_val = None, None

    model = SentimentLSTM(input_size=input_size, hidden_size=hidden_size)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    Xt = torch.tensor(X_fit)
    yt = torch.tensor(y_fit)

    best_val_loss, best_state, epochs_without_improve, stopped_at = float("inf"), None, 0, epochs
    train_loss_history, val_loss_history = [], []

    for epoch in range(epochs):
        model.train()
        opt.zero_grad()
        loss = loss_fn(model(Xt), yt)
        loss.backward()
        opt.step()

        model.eval()
        with torch.no_grad():
            train_loss_history.append(loss_fn(model(Xt), yt).item())

        if X_val is not None:
            with torch.no_grad():
                val_loss = loss_fn(model(torch.tensor(X_val)), torch.tensor(y_val)).item()
            val_loss_history.append(val_loss)
            if val_loss < best_val_loss:
                best_val_loss, epochs_without_improve = val_loss, 0
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
            else:
                epochs_without_improve += 1
                if epochs_without_improve >= patience:
                    stopped_at = epoch + 1
                    break

    if best_state is not None:
        model.load_state_dict(best_state)
    model._stopped_at_epoch = stopped_at
    model._train_loss_history = train_loss_history
    model._val_loss_history = val_loss_history
    model._train_var = float(np.var(y_fit))
    model._val_var = float(np.var(y_val)) if X_val is not None else None
    return model

def predict_lstm(model, X):
    model.eval()
    with torch.no_grad():
        return model(torch.tensor(X)).numpy()

def naive_persistence_forecast(y_train, n_test):
    return np.full(n_test, y_train[-1])

def select_arima_order(y_train, candidate_orders=((1, 0, 0), (1, 0, 1))):
    best_order, best_aic = (1, 0, 0), float("inf")
    for order in candidate_orders:
        try:
            fit = ARIMA(y_train, order=order).fit()
            if fit.aic < best_aic:
                best_aic, best_order = fit.aic, order
        except Exception:
            continue
    return best_order

def fit_arima_baseline(y_train, n_test, order=None):
    if order is None:
        order = select_arima_order(y_train)
    try:
        model = ARIMA(y_train, order=order).fit()
        return np.asarray(model.forecast(steps=n_test)), order
    except Exception as e:
        print(f"ARIMA failed ({e}) — falling back to naive baseline")
        return naive_persistence_forecast(y_train, n_test), order

## Build the wide multi-topic feature table (per outlet, per sentiment method)

In [8]:
sentiment_cache = {}  # (outlet, method) -> {topic: monthly Series}, so each method is only computed once

def score_topic_monthly(outlet, topic, method_name):
    cache_key = (outlet, method_name)
    if cache_key not in sentiment_cache:
        sentiment_cache[cache_key] = {}
    if topic in sentiment_cache[cache_key]:
        return sentiment_cache[cache_key][topic]

    df = datasets.get((outlet, topic))
    if df is None or len(df) < 20:
        sentiment_cache[cache_key][topic] = None
        return None

    scores = SENTIMENT_METHODS[method_name](df["text"].tolist(), topic)
    monthly = pd.Series(scores, index=df["publish_date"]).resample(AGG_FREQ).mean()
    sentiment_cache[cache_key][topic] = monthly
    return monthly

def build_wide_features(outlet, method_name):
    topic_series = {}
    for topic in TOPICS:
        s = score_topic_monthly(outlet, topic, method_name)
        if s is not None:
            topic_series[f"{topic}_sent"] = s

    if not topic_series:
        return pd.DataFrame()

    wide = pd.concat(topic_series, axis=1)  # outer join by default -> union of months
    wide = wide.ffill()          # carry forward last known sentiment through gap months
    wide = wide.fillna(0.0)      # neutral for months before a topic even started

    merged = wide.join(macro[["econ_value", "growth", "unemployment", "cpi"]], how="inner")
    merged = merged.dropna(subset=["growth"]).sort_index()
    return merged

FEATURE_SETS = {
    "sentiment_only": [f"{t}_sent" for t in TOPICS],
    "sentiment_plus_macro": [f"{t}_sent" for t in TOPICS] + ["unemployment", "cpi"],
}

## Main loop — 3 sentiment methods x 2 feature sets x 3 outlets, predicting GDP growth

In [9]:
results = []
wide_cache = {}  # (outlet, method) -> merged wide df, reused across feature-set variants and the recession test below

for outlet in OUTLETS:
    for method_name in SENTIMENT_METHODS:
        print(f"\n=== {outlet} / sentiment method: {method_name} ===")
        merged_full = build_wide_features(outlet, method_name)
        wide_cache[(outlet, method_name)] = merged_full
        n_months = len(merged_full)
        print(f"Combined monthly series (union of brexit/covid/energy_crisis coverage): {n_months} months")
        if n_months < 12:
            print("Fewer than 12 months — skipping.")
            continue

        split_idx = int(n_months * (1 - TEST_FRACTION))
        y = merged_full["growth"].values

        for feature_set_name, cols in FEATURE_SETS.items():
            X = merged_full[cols].values  # X is a table of numbers: one row per month, one column per feature

            # Ridge is sensitive to how big or small each feature's numbers are. If one
            # feature is on a much bigger scale than another (e.g. CPI is ~130, but
            # sentiment scores are ~-1 to 1), Ridge's penalty unfairly favours the
            # smaller-scale feature. This only actually happens for the
            # "sentiment_plus_macro" feature set, because that is the only one that
            # mixes small-scale sentiment numbers with large-scale macro numbers
            # (unemployment, CPI). "sentiment_only" never had this problem, so we
            # leave it completely untouched below.
            X_for_ridge = X  # by default, just use the same numbers as before

            if feature_set_name == "sentiment_plus_macro":
                # FEATURE_SETS["sentiment_only"] is just the list of sentiment column
                # names, so its length tells us how many of X's columns (starting from
                # the left) are sentiment columns, before the macro columns begin.
                n_sent = len(FEATURE_SETS["sentiment_only"])

                # split X into two separate pieces: the sentiment columns (which we will
                # leave alone) and the macro columns, unemployment and CPI (which need
                # to be rescaled)
                X_sent = X[:, :n_sent]
                X_macro = X[:, n_sent:]

                # StandardScaler rescales a column of numbers so that, on average, it
                # has a mean of 0 and a "typical spread" (standard deviation) of 1 —
                # this puts unemployment and CPI on the same footing as the sentiment
                # scores, so Ridge treats every feature fairly. We calculate this
                # rescaling rule using ONLY the training rows (X_macro[:split_idx]) —
                # never the test rows — so no information from the future leaks into
                # how the rescaling itself is defined.
                macro_scaler = StandardScaler().fit(X_macro[:split_idx])

                # now apply that same rescaling rule to ALL the macro data — both the
                # training rows and the test rows — using the rule we learned from
                # training only
                X_macro_scaled = macro_scaler.transform(X_macro)

                # glue the untouched sentiment columns and the newly-rescaled macro
                # columns back together, side by side, into one feature table again
                X_for_ridge = np.hstack([X_sent, X_macro_scaled])

            # train Ridge using X_for_ridge — for "sentiment_only" this is identical to
            # the original X; for "sentiment_plus_macro" this is the version with the
            # macro columns correctly rescaled
            alpha, _ = tune_ridge_alpha(X_for_ridge[:split_idx], y[:split_idx])
            m = Ridge(alpha=alpha).fit(X_for_ridge[:split_idx], y[:split_idx])
            pred = m.predict(X_for_ridge[split_idx:])
            results.append({"outlet": outlet, "sentiment_method": method_name, "feature_set": feature_set_name,
                             "model": "Ridge", **evaluate(y[split_idx:], pred), "n_months": n_months,
                             "tuned_param": f"alpha={alpha}"})

            # multivariate LSTM over the same feature columns — unchanged, still uses
            # the original unscaled X, since the LSTM was not the part that was broken
            series = X.astype(np.float32)
            growth_series = y.astype(np.float32)
            if n_months - LSTM_LOOKBACK < 12:
                print(f"  [{feature_set_name}] not enough months for LSTM lookback — skipping.")
            else:
                Xs, ys = [], []
                for i in range(len(series) - LSTM_LOOKBACK):
                    Xs.append(series[i:i + LSTM_LOOKBACK])
                    ys.append(growth_series[i + LSTM_LOOKBACK])
                Xs, ys = np.array(Xs), np.array(ys)
                split_seq = int(len(Xs) * (1 - TEST_FRACTION))

                y_scaler = StandardScaler()
                y_train_scaled = y_scaler.fit_transform(ys[:split_seq].reshape(-1, 1)).flatten()
                lstm_model = train_lstm(Xs[:split_seq], y_train_scaled, input_size=len(cols))
                pred3_scaled = predict_lstm(lstm_model, Xs[split_seq:])
                pred3 = y_scaler.inverse_transform(pred3_scaled.reshape(-1, 1)).flatten()
                results.append({"outlet": outlet, "sentiment_method": method_name, "feature_set": feature_set_name,
                                 "model": "Multivariate-LSTM", **evaluate(ys[split_seq:], pred3), "n_months": len(Xs),
                                 "tuned_param": f"stopped_epoch={lstm_model._stopped_at_epoch}"})

        # baselines are feature-independent — one per outlet/method combo, not per feature-set
        naive_pred = naive_persistence_forecast(y[:split_idx], len(y) - split_idx)
        results.append({"outlet": outlet, "sentiment_method": method_name, "feature_set": "n/a (news-free)",
                         "model": "Naive (no-change)", **evaluate(y[split_idx:], naive_pred), "n_months": n_months, "tuned_param": ""})
        arima_pred, arima_order = fit_arima_baseline(y[:split_idx], len(y) - split_idx)
        results.append({"outlet": outlet, "sentiment_method": method_name, "feature_set": "n/a (news-free)",
                         "model": "ARIMA(auto)", **evaluate(y[split_idx:], arima_pred), "n_months": n_months,
                         "tuned_param": f"order={arima_order}"})

results_df = pd.DataFrame(results)
print("\n\n=== FINAL RESULTS (all-topics, predicting GDP growth) ===")
print(results_df)
results_df.to_csv(f"{DATA_DIR}/model_results_all_topics_TEST.csv", index=False)
print(f"\nSaved to {DATA_DIR}/model_results_all_topics_TEST.csv")



=== bbc / sentiment method: whole_article ===


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Combined monthly series (union of brexit/covid/energy_crisis coverage): 168 months

=== bbc / sentiment method: aspect_based ===
Combined monthly series (union of brexit/covid/energy_crisis coverage): 168 months

=== bbc / sentiment method: ensemble ===
Combined monthly series (union of brexit/covid/energy_crisis coverage): 168 months

=== dailymail / sentiment method: whole_article ===
Combined monthly series (union of brexit/covid/energy_crisis coverage): 171 months

=== dailymail / sentiment method: aspect_based ===
Combined monthly series (union of brexit/covid/energy_crisis coverage): 171 months

=== dailymail / sentiment method: ensemble ===
Combined monthly series (union of brexit/covid/energy_crisis coverage): 171 months

=== guardian / sentiment method: whole_article ===
Combined monthly series (union of brexit/covid/energy_crisis coverage): 172 months

=== guardian / sentiment method: aspect_based ===
Combined monthly series (union of brexit/covid/energy_crisis coverage): 172

## Recession-specific test — both 2020 COVID and 2023 H2 windows

Same lead-up/recession/calm design as the main and energy_crisis notebooks, run
for every (outlet, sentiment_method, feature_set) combination, reusing the wide
tables built in the main loop above (run that cell first).

In [10]:
recession_results = []

for outlet in OUTLETS:
    for method_name in SENTIMENT_METHODS:
        merged_full = wide_cache.get((outlet, method_name))
        if merged_full is None or merged_full.empty:
            continue

        for feature_set_name, cols in FEATURE_SETS.items():
            for window in RECESSION_WINDOWS:
                rec_start, rec_end, label = window["start"], window["end"], window["name"]

                train = merged_full[merged_full.index < rec_start]
                leadup_start = rec_start - pd.DateOffset(months=LEADUP_MONTHS)
                test_leadup = merged_full[(merged_full.index >= leadup_start) & (merged_full.index < rec_start)]
                test_recession = merged_full[(merged_full.index >= rec_start) & (merged_full.index <= rec_end)]
                calm_end = rec_end + pd.DateOffset(months=CALM_COMPARISON_MONTHS)
                test_calm = merged_full[(merged_full.index > rec_end) & (merged_full.index <= calm_end)]

                tag = f"{outlet}/{method_name}/{feature_set_name} [{label}]"
                if len(train) < 24 or len(test_recession) < 2:
                    print(f"{tag}: {len(train)} training months — not enough data, skipping")
                    continue

                # Same idea as the main results loop above: only rescale the macro
                # columns (unemployment, CPI), and only when this particular feature
                # set actually includes them. "sentiment_only" is left completely
                # unscaled, exactly as it was before.
                train_vals = train[cols].values

                if feature_set_name == "sentiment_plus_macro":
                    # how many of these columns are sentiment columns, before the
                    # macro columns start
                    n_sent = len(FEATURE_SETS["sentiment_only"])

                    # learn the rescaling rule (mean/spread) using ONLY the training
                    # rows for this recession window — never the lead-up, recession,
                    # or calm test rows
                    macro_scaler = StandardScaler().fit(train_vals[:, n_sent:])

                    # rebuild the training feature table: sentiment columns unchanged,
                    # macro columns rescaled using the rule we just learned
                    train_X = np.hstack([train_vals[:, :n_sent], macro_scaler.transform(train_vals[:, n_sent:])])
                else:
                    macro_scaler = None   # no macro columns in this feature set, nothing to rescale
                    train_X = train_vals  # use the numbers exactly as they were

                alpha_r, _ = tune_ridge_alpha(train_X, train["growth"].values)
                m = Ridge(alpha=alpha_r).fit(train_X, train["growth"].values)
                print(f"{tag}: {len(train)} training months, tuned alpha={alpha_r}")

                for period_name, test_set in [
                    ("lead-up (pre-recession)", test_leadup),
                    ("recession", test_recession),
                    ("calm (post-recession)", test_calm),
                ]:
                    if len(test_set) < 2:
                        continue

                    # apply the exact same treatment to each test period that we
                    # applied to the training data: same column split, and the SAME
                    # rescaling rule learned from training (never re-learned from
                    # the test period itself, which would leak information)
                    test_vals = test_set[cols].values
                    if feature_set_name == "sentiment_plus_macro":
                        n_sent = len(FEATURE_SETS["sentiment_only"])
                        test_vals = np.hstack([test_vals[:, :n_sent], macro_scaler.transform(test_vals[:, n_sent:])])

                    pred = m.predict(test_vals)
                    recession_results.append({
                        "outlet": outlet, "sentiment_method": method_name, "feature_set": feature_set_name,
                        "recession": label, "period": period_name,
                        **evaluate(test_set["growth"].values, pred),
                        "n_months": len(test_set), "tuned_param": f"alpha={alpha_r}",
                    })

recession_df = pd.DataFrame(recession_results)
print("\n\n=== LEAD-UP vs RECESSION vs CALM COMPARISON (all-topics, Ridge) ===")
print(recession_df)
recession_df.to_csv(f"{DATA_DIR}/recession_specific_results_all_topics_TEST.csv", index=False)
print(f"\nSaved to {DATA_DIR}/recession_specific_results_all_topics_TEST.csv")


bbc/whole_article/sentiment_only [2020 COVID recession]: 91 training months, tuned alpha=30.0
bbc/whole_article/sentiment_only [2023 H2 recession]: 133 training months, tuned alpha=30.0
bbc/whole_article/sentiment_plus_macro [2020 COVID recession]: 91 training months, tuned alpha=30.0
bbc/whole_article/sentiment_plus_macro [2023 H2 recession]: 133 training months, tuned alpha=30.0
bbc/aspect_based/sentiment_only [2020 COVID recession]: 91 training months, tuned alpha=30.0
bbc/aspect_based/sentiment_only [2023 H2 recession]: 133 training months, tuned alpha=30.0
bbc/aspect_based/sentiment_plus_macro [2020 COVID recession]: 91 training months, tuned alpha=30.0
bbc/aspect_based/sentiment_plus_macro [2023 H2 recession]: 133 training months, tuned alpha=30.0
bbc/ensemble/sentiment_only [2020 COVID recession]: 91 training months, tuned alpha=0.1
bbc/ensemble/sentiment_only [2023 H2 recession]: 133 training months, tuned alpha=10.0
bbc/ensemble/sentiment_plus_macro [2020 COVID recession]: 91 

## Clear, separated results tables

In [11]:
print("=== MAIN RESULTS, by outlet / sentiment method / feature set ===\n")
for (outlet, method, fs), sub in results_df.groupby(["outlet", "sentiment_method", "feature_set"]):
    print(f"--- {outlet} / {method} / {fs} ---")
    print(sub[["model", "rmse", "mae", "r2", "directional_accuracy", "n_months", "tuned_param"]].to_string(index=False))
    print()

if len(recession_df):
    print("\n=== RECESSION-SPECIFIC RESULTS, by outlet / sentiment method / feature set / recession ===\n")
    for (outlet, method, fs, rec), sub in recession_df.groupby(["outlet", "sentiment_method", "feature_set", "recession"]):
        print(f"--- {outlet} / {method} / {fs} / {rec} ---")
        print(sub[["period", "rmse", "mae", "r2", "directional_accuracy", "n_months", "tuned_param"]].to_string(index=False))
        print()
else:
    print("\nNo combination had 24+ pre-recession training months for the recession-specific test.")

=== MAIN RESULTS, by outlet / sentiment method / feature set ===

--- bbc / aspect_based / n/a (news-free) ---
            model     rmse      mae        r2  directional_accuracy  n_months     tuned_param
Naive (no-change) 0.539283 0.488134 -4.522067              0.294118       168                
      ARIMA(auto) 0.238077 0.202504 -0.076219              0.558824       168 order=(1, 0, 1)

--- bbc / aspect_based / sentiment_only ---
            model     rmse      mae        r2  directional_accuracy  n_months      tuned_param
            Ridge 0.237529 0.202156 -0.071277              0.529412       168       alpha=30.0
Multivariate-LSTM 0.249028 0.209272 -0.167092              0.545455       165 stopped_epoch=49

--- bbc / aspect_based / sentiment_plus_macro ---
            model     rmse      mae        r2  directional_accuracy  n_months      tuned_param
            Ridge 0.282805 0.238536 -0.518596              0.529412       168       alpha=30.0
Multivariate-LSTM 0.233557 0.196161 